## 1. Import Libraries

In [1]:
# ============================================================
# FARMER REGISTRATION - DATA CLEANING & VALIDATION
# ============================================================

import pandas as pd
import numpy as np
import re

## 2. Load Raw Dataset

In [2]:
# ============================================================
# LOAD RAW DATA
# ============================================================

df = pd.read_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/01_raw/farmer_registration_raw.csv")

print("Dataset Loaded Successfully")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.head()

Dataset Loaded Successfully
Rows: 1020
Columns: 33


,number,formid,form.section_1_farmer_identification.community,form.section_1_farmer_identification.first_name,form.section_1_farmer_identification.last_name,form.section_1_farmer_identification.gender,form.section_1_farmer_identification.date_of_birth,form.section_1_farmer_identification.national_id,form.section_1_farmer_identification.phone_number,form.section_1_farmer_identification.farmer_photograph,...,form.section_5_livestock.livestock_and_number_kept_integer.number_of_rabbits,form.section_6_farm_location.farm_gps_location,form.section_7_consent.farmer_accepts_or_decline_to_participate_in_the_programme.choice,form.case.@case_id,completed_time,started_time,username,received_on,form_link,hq_user
0,1,5daea065-afd1-444e-9b2d-44e33c8747b9,community_05,Amina,Balogun,male,1994-07-28,NaN,8055002657,https://www.commcarehq.org/photos/farmer_1.jpg,...,---,10.3435918 7.3375823 0.0 20.0,yes,b444a62e-13c0-4ccd-8ec2-838bb2dfe5bd,2026-01-12 00:07:00,2026-01-12,tester_05,2026-01-12 00:07:19,https://www.commcarehq.org/form/5daea065-afd1-...,tester_05
1,2,d0cf4629-b9a2-4991-af86-259532474a48,community_04,Elizabeth,David,male,1987-01-07,2.161605e+10,8002202126,https://www.commcarehq.org/photos/farmer_2.jpg,...,---,10.5371408 7.5104 0.0 20.0,yes,12f1f5af-5256-4e18-96cf-f7a1dac16e7d,2026-01-09 00:08:00,2026-01-09,tester_04,2026-01-09 00:08:15,https://www.commcarehq.org/form/d0cf4629-b9a2-...,tester_04
2,3,606344d1-cc32-46d3-b6ce-16e5d3a80216,community_03,Fatima,Daniel,female,1981-03-26,NaN,8039422525,https://www.commcarehq.org/photos/farmer_3.jpg,...,---,10.38722 7.3005656 0.0 20.0,yes,0b061fab-6c5b-44bb-90cf-6f440d76f65d,2026-01-03 00:07:00,2026-01-03,tester_03,2026-01-03 00:07:08,https://www.commcarehq.org/form/606344d1-cc32-...,tester_03
3,4,d772635c-2543-44f9-9de5-6cadccf94f19,community_05,Mercy,Lawal,male,1985-02-11,1.194873e+10,8098251982,https://www.commcarehq.org/photos/farmer_4.jpg,...,---,10.2231701 7.5512038 0.0 20.0,yes,e0bd8266-2f5c-49e1-bcce-1dc843a1c529,2026-01-22 00:07:00,2026-01-22,tester_05,2026-01-22 00:07:20,https://www.commcarehq.org/form/d772635c-2543-...,tester_05
4,5,d1ecec10-268b-4b12-be46-0025a7150848,community_06,Emmanuel,Sani,female,2000-09-29,6.843988e+10,8099767363,https://www.commcarehq.org/photos/farmer_5.jpg,...,---,10.3941658 7.4156936 0.0 20.0,yes,f6d3f342-1621-49ae-826b-6d3a9cfcec86,2026-05-27 00:06:00,2026-05-27,tester_06,2026-05-27 00:06:20,https://www.commcarehq.org/form/d1ecec10-268b-...,tester_06


## 3. Initial Data Profiling

In [3]:
# ============================================================
# INITIAL DATA PROFILING
# ============================================================

print("\nData Types\n")
print(df.dtypes)

print("\n")

print("Missing Values\n")
print(df.isnull().sum())

print("\n")

print("Duplicate Rows:", df.duplicated().sum())


Data Types

number                                                                                                  int64
formid                                                                                                 object
form.section_1_farmer_identification.community                                                         object
form.section_1_farmer_identification.first_name                                                        object
form.section_1_farmer_identification.last_name                                                         object
form.section_1_farmer_identification.gender                                                            object
form.section_1_farmer_identification.date_of_birth                                                     object
form.section_1_farmer_identification.national_id                                                      float64
form.section_1_farmer_identification.phone_number                                                       int

## 4. Remove Duplicate Registrations

In [4]:
# ============================================================
# REMOVE DUPLICATE REGISTRATIONS
# ============================================================

rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

duplicates_removed = rows_before - rows_after

print(f"Duplicate records removed: {duplicates_removed}")

print(f"Remaining records: {rows_after}")

Duplicate records removed: 20
Remaining records: 1000


## 5. Standardise Phone Numbers

In [5]:
# ============================================================
# STANDARDISE PHONE NUMBERS
# ============================================================

# Name of the phone number column in the dataset
phone_column = "form.section_1_farmer_identification.phone_number"

# Function to clean and standardise phone numbers
def clean_phone(phone):

    # Return missing values unchanged
    if pd.isna(phone):
        return np.nan

    # Convert to string and remove leading/trailing spaces
    phone = str(phone).strip()

    # Remove all non-numeric characters (spaces, dashes, brackets, etc.)
    phone = re.sub(r"\D", "", phone)

    # Convert Nigerian international format (234...) to local format (0...)
    if phone.startswith("234"):
        phone = "0" + phone[3:]

    # Add a leading zero if the number has only 10 digits
    elif len(phone) == 10:
        phone = "0" + phone

    # Return the cleaned phone number
    return phone


# Apply the cleaning function to the phone number column
df[phone_column] = df[phone_column].apply(clean_phone)

# Confirmation message
print("Phone numbers standardised.")

Phone numbers standardised.


## 6. Standardise Name Capitalization

In [6]:
# ============================================================
# STANDARDISE NAME CAPITALIZATION
# ============================================================

first_name = "form.section_1_farmer_identification.first_name"

last_name = "form.section_1_farmer_identification.last_name"

df[first_name] = df[first_name].str.title()

df[last_name] = df[last_name].str.title()

print("Name capitalization standardised.")

Name capitalization standardised.


## 7. Validate GPS

In [7]:
# ============================================================
# GPS VALIDATION
# ============================================================

gps_column = "form.section_6_farm_location.farm_gps_location"

def gps_valid(gps):

    try:

        latitude = float(gps.split()[0])
        longitude = float(gps.split()[1])

        return (
            10.20 <= latitude <= 10.60
            and
            7.20 <= longitude <= 7.60
        )

    except:

        return False


df["gps_valid"] = df[gps_column].apply(gps_valid)

print(df["gps_valid"].value_counts())

gps_valid
True    1000
Name: count, dtype: int64


## 8. Final Validation Summary

In [8]:
# ============================================================
# FINAL VALIDATION SUMMARY
# ============================================================

print("Rows:", len(df))

print("\n")

print("Missing Values")

print(df.isnull().sum())

print("\n")

print("Duplicate Rows")

print(df.duplicated().sum())

print("\n")

print("GPS Validation")

print(df["gps_valid"].value_counts())

Rows: 1000


Missing Values
number                                                                                                  0
formid                                                                                                  0
form.section_1_farmer_identification.community                                                          0
form.section_1_farmer_identification.first_name                                                         0
form.section_1_farmer_identification.last_name                                                          0
form.section_1_farmer_identification.gender                                                             0
form.section_1_farmer_identification.date_of_birth                                                      0
form.section_1_farmer_identification.national_id                                                      329
form.section_1_farmer_identification.phone_number                                                       0
form.section_1_far

## 9. Export Clean Dataset

In [9]:
# ============================================================
# EXPORT CLEAN DATASET
# ============================================================

df.to_csv("C:/Users/HP PC/Documents/jacob_git_projects/Monitoring-and-Evaluation-System/dataset/02_cleaned/farmer_registration_clean.csv",
          index=False)

print("Clean dataset exported successfully.")

Clean dataset exported successfully.
